# 🎯 RENTAP-1: Multi-Frame Drone Tracking Benchmark & Evaluation

Notebook ini direka khas untuk menguji dan menganalisis prestasi **Tracking Dron (Multi-Frame Video)** berbanding ujian biasa **Detection (Single-Frame Static)**.

### 📌 Perbezaan Utama:
| Aspek | Detection (1 Frame) | Tracking (Multi-Frame) |
| :--- | :--- | :--- |
| **Input** | Imej pegun rawak (`.jpg`, `.png`) | Urutan video berterusan (`.mp4`, webcam feed) |
| **Fokus Utama** | Mengesan kehadiran dron & koordinat kotak | Mengekalkan identiti (Track ID), arah pergerakan & momentum sasaran |
| **Metrik Utama** | Precision, Recall, mAP@50, mAP@50-95 | **ID Switches (IDSW)**, **Lock Retention Rate (%)**, **Trajectory Smoothness**, **Real-Time FPS** |
| **Peranan Sentry** | Tahu dron wujud | Menggerakkan gimbal/servo mengikut sasaran secara lancar tanpa gegaran |

In [ ]:
# 1. Setup & Import Modul
import os
import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# Import enjin AI Vision RENTAP-1
from core.ai_vision import TurretVision

print("✅ Modul berjaya diimport!")

In [ ]:
# 2. Konfigurasi Fail Model & Sumber Video
# Pilih model YOLO (.pt) dan video dron untuk diuji
MODEL_PATH = "models/best (2).pt"        # Model aktif YOLOv8s
VIDEO_SOURCE = "test_drone_flight.mp4"   # Tukar kepada nama fail video anda atau 0 untuk webcam
OUTPUT_VIDEO = "output_tracking_eval.mp4"

# Inisialisasi Turret Vision (Dual-Threshold Hysteresis + Centroid/IoU Association + EMA Smoothing)
vision = TurretVision(model_path=MODEL_PATH)
print(f"🚀 TurretVision dimuatkan dengan model: {MODEL_PATH}")

In [ ]:
# 3. Enjin Penilaian Multi-Frame Tracking
def run_tracking_evaluation(video_path, max_frames=500):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"⚠️ Gagal membuka {video_path}. Sila pastikan fail wujud atau gunakan webcam (0).")
        return None

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_in = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_vid_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"📹 Membaca video: {w}x{h} @ {fps_in:.1f} FPS (Jumlah frame: {total_vid_frames})")

    # Metrik Pengesanan Multi-Frame
    history_tracks = defaultdict(list)    # track_id -> [(frame_num, cx, cy, conf, w_box, h_box)]
    frame_fps_list = []
    frame_detected_list = []
    frames_processed = 0
    
    start_eval_time = time.time()

    while True:
        ret, frame = cap.read()
        if not ret or (max_frames and frames_processed >= max_frames):
            break

        frames_processed += 1
        t0 = time.time()

        # Jalankan tracking pada frame semasa
        annotated, boxes = vision.scan_frame(frame)
        t1 = time.time()

        fps_instant = 1.0 / max(0.001, (t1 - t0))
        frame_fps_list.append(fps_instant)

        if len(boxes) > 0:
            frame_detected_list.append(1)
            for b in boxes:
                x1, y1, x2, y2, tid, conf, c_name = b[:7]
                cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
                bw, bh = (x2 - x1), (y2 - y1)
                history_tracks[tid].append({
                    "frame": frames_processed,
                    "cx": cx,
                    "cy": cy,
                    "conf": conf,
                    "bw": bw,
                    "bh": bh
                })
        else:
            frame_detected_list.append(0)

    cap.release()
    total_time = time.time() - start_eval_time
    
    results = {
        "total_frames": frames_processed,
        "total_time": total_time,
        "avg_fps": frames_processed / max(0.001, total_time),
        "fps_list": frame_fps_list,
        "detection_ratio": sum(frame_detected_list) / max(1, frames_processed) * 100.0,
        "tracks": history_tracks,
        "unique_ids": list(history_tracks.keys()),
        "frame_width": w,
        "frame_height": h
    }
    return results

print("✅ Fungsi penilaian siap didaftarkan!")

In [ ]:
# 4. Jalankan Ujian & Paparkan Metrik Utama
# Jika anda tiada fail video, fungsi ini boleh dijalankan pada webcam dengan meletakkan 0
results = None
if os.path.exists(VIDEO_SOURCE) or str(VIDEO_SOURCE) == '0':
    results = run_tracking_evaluation(VIDEO_SOURCE, max_frames=300)
else:
    print(f"ℹ️ Fail '{VIDEO_SOURCE}' belum wujud dalam direktori.")
    print("💡 Anda boleh meletakkan sebarang fail video rakaman dron (.mp4) di dalam folder projek dan kemas kini VIDEO_SOURCE.")

if results:
    print("\n========== 📊 LAPORAN KEPUTUSAN TRACKING ==========")
    print(f"• Jumlah Bingkai Diproses : {results['total_frames']} frame")
    print(f"• Purata Kelajuan (FPS)   : {results['avg_fps']:.2f} FPS")
    print(f"• Lock Retention Rate     : {results['detection_ratio']:.2f}%")
    print(f"• Jumlah Unique Track ID  : {len(results['unique_ids'])}")
    print(f"• Senarai Track ID        : {results['unique_ids']}")
    print("===================================================")

In [ ]:
# 5. Visualisasi Analisis Graf (Trajectory, ID Persistence & FPS)
if results and len(results['tracks']) > 0:
    plt.figure(figsize=(16, 10))

    # Graf 1: 2D Spatial Trajectory (Laluan Pergerakan Dron dalam Skrin)
    plt.subplot(2, 2, 1)
    for tid, pts in results['tracks'].items():
        xs = [p['cx'] for p in pts]
        ys = [p['cy'] for p in pts]
        plt.plot(xs, ys, marker='o', markersize=3, label=f'ID #{tid} ({len(pts)} frames)')
    plt.xlim(0, results['frame_width'])
    plt.ylim(results['frame_height'], 0)  # Invert Y mengikut orientasi koordinat imej
    plt.title("🎯 2D Drone Flight Trajectory (X vs Y Coordinates)")
    plt.xlabel("X Pixel")
    plt.ylabel("Y Pixel")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()

    # Graf 2: ID Consistency / Lifespan (Adakah berlaku ID Switches)
    plt.subplot(2, 2, 2)
    for tid, pts in results['tracks'].items():
        frames_active = [p['frame'] for p in pts]
        id_vals = [tid] * len(frames_active)
        plt.scatter(frames_active, id_vals, label=f'ID #{tid}', s=15)
    plt.title("🆔 Track ID Persistence Across Time (ID Switches Check)")
    plt.xlabel("Frame Number")
    plt.ylabel("Active Track ID")
    plt.grid(True, linestyle="--", alpha=0.6)

    # Graf 3: Real-Time FPS Stability
    plt.subplot(2, 2, 3)
    plt.plot(results['fps_list'], color='forestgreen', alpha=0.8)
    plt.axhline(y=results['avg_fps'], color='red', linestyle='--', label=f"Purata: {results['avg_fps']:.1f} FPS")
    plt.title("⚡ Processing Speed Stability (FPS Over Time)")
    plt.xlabel("Frame Number")
    plt.ylabel("FPS")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()

    # Graf 4: Confidence Score Distribution over Time
    plt.subplot(2, 2, 4)
    for tid, pts in results['tracks'].items():
        frames_active = [p['frame'] for p in pts]
        confs = [p['conf'] for p in pts]
        plt.plot(frames_active, confs, label=f'ID #{tid} Conf')
    plt.title("📈 Target Confidence Score Consistency")
    plt.xlabel("Frame Number")
    plt.ylabel("Confidence (0.0 - 1.0)")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Sila jalankan penilaian dengan video untuk menjana graf metrik.")